# Sanity Check - one full epoch of the marker pipeline

Runs the **real** `src/` modules end-to-end on a small data slice so failures
surface in ~2 minutes instead of after a full 5-fold run.

**How to use**

1. Attach the **Compartment** dataset as an input (provides `dataset/` + `trial/`).
2. Add the repo via **Add Input -> GitHub** (or set `USE_KAGGLE_INPUT=False` +
   `GITHUB_URL`). 
3. Click **Run All** (GPU runtime).

**What is verified**

- marker tokens are recognized and the embedding table matches the vocab
- forward pass shapes / finiteness
- `CombinedLoss` (MSE+CCC) and the ranking variant
- Phase-1 optimizer gradient groups (embeddings train, encoder frozen, heads train)
- gradient accumulation: scheduler steps correctly (`accum_steps` optimizer steps per epoch)
- `Trainer.fit` end-to-end: Phase 1 -> Phase 2 switch, checkpoint save, `FoldResult`
- checkpoint round-trip: reloaded model reproduces the val predictions
- `run_predict`: trial + TSV submission from the slice checkpoint
- optional: full-data `main.py train80` for exactly 2 epochs (real-perf check)

## 1. Locate / clone the repository

In [ ]:
import glob, os, subprocess, sys, zipfile

# ==== 1. Point at your repo ==========================================
USE_KAGGLE_INPUT = True   # True  -> GitHub repo mounted via "Add Input -> GitHub"
                          # False -> git clone the URL below into /kaggle/working
GITHUB_URL = 'https://github.com/AmnO-O/MoTune.git'   # <-- EDIT ME

def sh(cmd, cwd=None):
    print('>>>', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    sys.stdout.write(r.stdout)
    sys.stderr.write(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')

REPO = None
if USE_KAGGLE_INPUT:
    mains = sorted(glob.glob('/kaggle/input/github/**/main.py', recursive=True))
    if mains:
        REPO = os.path.dirname(mains[0])   # read-only snapshot; Kaggle input is fixed
if REPO is None:
    REPO = '/kaggle/working/compartment'
    if os.path.isdir(os.path.join(REPO, '.git')):
        # in-session clone from an earlier run -> refresh to the latest commit
        sh(f'git -C {REPO} fetch --depth 1 origin main')
        sh(f'git -C {REPO} reset --hard origin/main')
    elif not os.path.isdir(os.path.join(REPO, 'src')):
        sh(f'git clone --depth 1 {GITHUB_URL} {REPO}')

print('REPO =', REPO)
print('commit =', __import__('subprocess').run(
    ['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
    capture_output=True, text=True).stdout.strip())
os.chdir(REPO)
sys.path.insert(0, REPO)
print('CWD =', os.getcwd())

## 2. Environment
torch / transformers come pre-installed on Kaggle; only install if missing.

In [ ]:
import subprocess, sys

check = subprocess.run(
    [sys.executable, '-c', 'import torch, transformers, pandas, numpy, sklearn, scipy'],
    capture_output=True,
)
if check.returncode != 0:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch', 'transformers>=4.41', 'pandas', 'numpy', 'scikit-learn', 'scipy'],
        check=True,
    )
    print('Installed missing packages.')
else:
    print('Environment OK (torch, transformers, pandas, numpy, sklearn, scipy).')

## 3. Settings & data slice
All checks below run on a 64-row slice through the **real** pipeline objects
(`Config`, `NNDataset`, `build_model`, `Trainer.fit`, `run_predict`).

In [ ]:
import glob, json, os, sys
from pathlib import Path

# ==== sanity settings =================================================
SLICE_ROWS   = 64        # train rows used for the in-notebook checks
BATCH_SIZE   = 16
ACCUM_STEPS  = 2            # exercises the gradient-accumulation path
NUM_EPOCHS   = 2            # epoch 1 = FROZEN (Phase 1), epoch 2 = UNFROZEN (Phase 2)
FREEZE_EPOCHS = 1           # so the phase switch really happens
LAMBDA_RANK  = 1.0          # turn on the ranking loss so it runs end-to-end
DO_FULL_CLI  = True         # ALSO run the full-data `main.py train80` for 2 epochs

# ==== paths ============================================================
def find_data_dir():
    if os.path.isfile('dataset/en-nn-train.tsv'):
        return os.getcwd()
    hits = sorted(glob.glob('/kaggle/input/**/dataset/en-nn-train.tsv', recursive=True))
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))   # parent holds dataset/ and trial/
    raise RuntimeError('Training data not found - attach the Compartment dataset input.')

DATA_DIR = find_data_dir()
OUT = Path('/kaggle/working/sanity_out' if os.path.isdir('/kaggle/working') else './sanity_out')
(OUT / 'models').mkdir(parents=True, exist_ok=True)
(OUT / 'submission').mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('OUT      =', OUT)

## 4. Config, device, model & marker checks

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.amp import GradScaler
from torch.utils.data import DataLoader

from config import Config
from src.constants import MARKER_TOKENS
from src.dataset import NNDataset
from src.loss import CombinedLoss
from src.model import build_model, embedding_table
from src.pipelines import load_tokenizer
from src.train import evaluate, train_epoch
from src.trainer import Trainer
from src.utils import get_device, get_logger, set_seed

cfg = Config.defaults().update(
    batch_size=BATCH_SIZE, accum_steps=ACCUM_STEPS,
    num_epochs=NUM_EPOCHS, freeze_epochs=FREEZE_EPOCHS,
    max_length=64, max_context_length=128,
    lambda_rank=LAMBDA_RANK, rank_margin=0.5, seed=42,
)
cfg.validate()

device = get_device()
logger = get_logger('sanity', log_dir=OUT)
set_seed(cfg.seed)
print('device:', device)

df = pd.read_csv(os.path.join(DATA_DIR, 'dataset', 'en-nn-train.tsv'), sep='\t')
train_df = df.iloc[:SLICE_ROWS].reset_index(drop=True)
val_df = df.iloc[SLICE_ROWS: 2 * SLICE_ROWS].reset_index(drop=True)
print(f'slice: train {len(train_df)} rows | val {len(val_df)} rows')

In [ ]:
tok = load_tokenizer(cfg)

# markers are single tokens inside real sentences
s = 'our <mwe><mod>flea</mod><head>market</head></mwe> is a <mwe><mod>night</mod><head>watch</head></mwe> story'
toks = tok.convert_ids_to_tokens(tok(s).input_ids)
for m in MARKER_TOKENS:
    assert m in tok.get_vocab(), f'missing from vocab: {m}'
    assert m in toks, f'marker split into subwords: {m} -> {toks}'
print('markers recognized:', [t for t in toks if t.startswith("<")])

model = build_model(cfg, tok, device)
emb_rows = embedding_table(model).weight.shape[0]
assert emb_rows == len(tok), (emb_rows, len(tok))
n_params = sum(p.numel() for p in model.parameters())
print(f'embedding table == vocab: {emb_rows} rows | trainable params: {n_params:,}')

In [ ]:
ds = NNDataset(train_df, tok, cfg.max_length, cfg.max_context_length)
loader = DataLoader(ds, batch_size=BATCH_SIZE)
batch = {k: v.to(device, non_blocking=True) for k, v in next(iter(loader)).items()}

model.eval()
with torch.no_grad():
    mod_pred, head_pred = model(batch)
assert mod_pred.shape == (BATCH_SIZE,) and head_pred.shape == (BATCH_SIZE,)
assert torch.isfinite(mod_pred).all() and torch.isfinite(head_pred).all()
print('forward OK | mod_head sample:', mod_pred[:3].tolist(), head_pred[:3].tolist())

In [ ]:
crit_base = CombinedLoss(ccc_weight=cfg.ccc_weight)
crit_rank = CombinedLoss(ccc_weight=cfg.ccc_weight,
                         lambda_rank=cfg.lambda_rank, rank_margin=cfg.rank_margin)

l_base = crit_base(mod_pred, batch['mod_avg'])
l_rank = crit_rank(mod_pred, batch['mod_avg'])
assert torch.isfinite(l_base) and torch.isfinite(l_rank)
print(f'loss (MSE+CCC): {l_base.item():.4f} | with ranking: {l_rank.item():.4f}')

In [ ]:
model2 = build_model(cfg, tok, device)          # fresh frozen model
trainer = Trainer(cfg, device, logger, OUT)

emb = embedding_table(model2).weight
enc0 = next(model2.bert.layers[0].parameters())
head0 = next(model2.mod_regressor.parameters())
assert not enc0.requires_grad and not emb.requires_grad, 'fresh build_model must be fully frozen'

opt = trainer._phase1(model2, 100)[0]        # turns embeddings ON for marker learning
assert emb.requires_grad and not enc0.requires_grad, 'phase1: embeddings train, encoder frozen'
assert head0.requires_grad, 'heads must be trainable in phase1'
model2.train(); opt.zero_grad()
b2 = {k: v.to(device, non_blocking=True) for k, v in
      next(iter(DataLoader(NNDataset(train_df, tok, cfg.max_length, cfg.max_context_length), batch_size=BATCH_SIZE))).items()}
mp2, hp2 = model2(b2)
loss = torch.nn.functional.mse_loss(mp2, b2['mod_avg']) + torch.nn.functional.mse_loss(hp2, b2['head_avg'])
loss.backward()

assert emb.grad is not None, 'embeddings should get grads (marker learning)'
assert enc0.grad is None, 'frozen encoder layer must NOT get grads'
assert head0.grad is not None, 'heads should get grads'
print('phase-1 grad groups OK: embeddings train, encoder frozen, heads train')

In [ ]:
n_micro = int(np.ceil(len(train_df) / BATCH_SIZE))
expected = {1: n_micro, ACCUM_STEPS: int(np.ceil(n_micro / ACCUM_STEPS))}
for acc, want in expected.items():
    m = build_model(cfg, tok, device)
    opt, sched = trainer._phase1(m, 1000)
    count = {'n': 0}                     # count scheduler.step() calls == optimizer updates
    _raw_sched_step = sched.step
    def _counting_step(*a, **k):
        count['n'] += 1
        return _raw_sched_step(*a, **k)
    sched.step = _counting_step
    _ = train_epoch(m, DataLoader(NNDataset(train_df, tok, cfg.max_length, cfg.max_context_length),
                                  batch_size=BATCH_SIZE), opt, sched,
                    crit_rank, GradScaler('cuda'), device,
                    grad_clip=cfg.grad_clip, accum_steps=acc)
    got = count['n']
    print(f'accum_steps={acc} -> optimizer steps {got} (expected {want})')
    assert got == want, f'gradient accumulation broken: got {got}, want {want}'
print('gradient accumulation OK')

## 5. Full `Trainer.fit` on the slice (runs Phase 1 + Phase 2)

In [ ]:
res = trainer.fit(train_df, val_df, tok, fold=None, ckpt_name='best.pt')

ckpt = OUT / 'models' / 'best.pt'
assert ckpt.is_file(), 'checkpoint not saved'
assert np.isfinite(res.rho_mean), res
assert len(res.history) == 2 and res.history[0]['phase'] == 'FROZEN' \
    and res.history[1]['phase'] == 'UNFROZEN-TOP'
print('FIT OK | epochs:', [h['epoch'] for h in res.history])
print('        phases:', [h['phase'] for h in res.history])
print('        best Mean rho %.4f at epoch %d' % (res.rho_mean, res.best_epoch))

## 6. Checkpoint round-trip + predict on the slice

In [ ]:
modelB = build_model(cfg, tok, device, dropout=0.0)
state = torch.load(ckpt, map_location=device, weights_only=True)
modelB.load_state_dict(state)
modelB.eval()

val_loader = DataLoader(NNDataset(val_df, tok, cfg.max_length, cfg.max_context_length),
                        batch_size=BATCH_SIZE)
mp_r, hp_r, _, _ = evaluate(modelB, val_loader, device)
d_mod = float(np.max(np.abs(mp_r - res.best_mod_pred)))
d_head = float(np.max(np.abs(hp_r - res.best_head_pred)))
assert d_mod < 1e-5 and d_head < 1e-5
print(f'checkpoint round-trip OK | max|d| mod={d_mod:.2e} head={d_head:.2e}')

In [ ]:
from src import pipelines

cfg_single = cfg.update(predict_mode='single', output_dir=str(OUT), data_path=DATA_DIR)
pred_metrics = pipelines.run_predict(cfg_single, logger, device, Path(DATA_DIR), OUT)

sub_path = OUT / 'submission' / 'en-nn-trial-pred.tsv'
assert sub_path.is_file(), 'submission file missing'
sub = pd.read_csv(sub_path, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
assert len(sub) == 10
print('predict OK | trial Mean rho %.4f | submission rows %d' % (pred_metrics['trial_rho_mean'], len(sub)))
print(sub.head(3).to_string(index=False))

## 7. Optional: full-data CLI run (exactly 2 epochs of `main.py train80`)

Exercises the real entry point on the full ~3.5k-row set (how the actual run
feels in speed / memory). Disable with `DO_FULL_CLI = False`.

In [ ]:
import subprocess, sys

if DO_FULL_CLI:
    out = OUT / 'cli_run'
    cmd = [sys.executable, 'main.py', 'train80',
           '--epochs', '2', '--freeze-epochs', '1',
           '--data-path', DATA_DIR, '--output-dir', str(out)]
    print('>>>', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    sys.stdout.write(r.stdout[-1200:])
    sys.stderr.write(r.stderr[-1200:])
    if r.returncode != 0:
        print('CLI run FAILED (rc=%d) - see log above' % r.returncode)
    else:
        mj = json.loads((out / 'metrics.json').read_text())
        print('CLI OK | val Mean rho %.4f at epoch %d' % (mj['val_rho_mean'], mj['best_epoch']))
else:
    print('DO_FULL_CLI=False - skipped the full-data run.')

## 8. Summary

In [ ]:
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'GPU memory: {free / 1e9:.2f} GB free / {total / 1e9:.2f} GB')
else:
    print('CPU device - no GPU memory to report.')

print()
print('ALL SANITY CHECKS PASSED.')
print()
print('Next step: if the CLI run above looked healthy, disable DO_FULL_CLI and run')
print('the real thing via notebook/run_main.ipynb (train80 first, then train5 + predict).')